# LizyML Tutorial: Multiclass Classification with LightGBM

This notebook demonstrates the full LizyML workflow on a **multiclass classification** task:

1. Data preparation (Vehicle Silhouettes dataset from UCI / OpenML)
2. Config setup (StratifiedKFold default, early stopping)
3. Model fit (5-fold stratified CV with early stopping)
4. Evaluate — metrics table + learning curve
5. ROC Curve (OvR, IS vs OOS)
6. Confusion Matrix (IS vs OOS)
7. Feature importance — Split, Gain, SHAP

**Dataset**: [Vehicle Silhouettes](https://www.openml.org/d/54) — 846 rows, 18 features, 4 classes  
**Target**: `vehicle_class` (0=bus, 1=opel, 2=saab, 3=van)

## 1. Setup

In [ ]:
from __future__ import annotations

import pandas as pd
from sklearn.datasets import fetch_openml

from lizyml import Model

## 2. Data Preparation

Download the [Vehicle Silhouettes dataset](https://www.openml.org/d/54) from OpenML.
This UCI dataset classifies vehicle types (bus, opel, saab, van) from 18 geometric
features extracted from silhouette images.

In [ ]:
vehicle = fetch_openml("vehicle", version=1, as_frame=True, parser="auto")

df = vehicle.data.copy()
df.columns = [c.lower().replace(".", "_") for c in df.columns]

classes = sorted(vehicle.target.unique())
label_map = {c: i for i, c in enumerate(classes)}
df["vehicle_class"] = vehicle.target.map(label_map).astype(int)

print(f"Shape: {df.shape}")
print(f"Classes: {label_map}")
print(f"Class distribution:\n{df['vehicle_class'].value_counts().sort_index()}")
df.head()

In [ ]:
df.describe()

## 3. Config

All 14 LightGBM parameters are explicitly specified as a tutorial reference:

- **model.params** (9): native LightGBM parameters — `objective`, `n_estimators`, `learning_rate`, `max_depth`, `feature_fraction`, `bagging_fraction`, `bagging_freq`, `lambda_l2`, `max_bin`
- **smart params** (3): data-size-relative — `num_leaves_ratio`, `min_data_in_leaf_ratio`, `min_data_in_bin_ratio` (resolved at fit time via `auto_num_leaves`)
- **training** (2): early stopping — `rounds`, `validation_ratio`

For multiclass classification, `split` is omitted — LizyML defaults to `stratified_kfold` (H-0013).
`balanced=True` computes `sample_weight` from class frequencies.

Multiclass metrics include OvR extensions for AUC, AUC-PR, and Brier (H-0018).

Vehicle dataset has 846 rows — using 3-fold CV to ensure sufficient training data per fold.

In [ ]:
config = {
    "config_version": 1,
    "task": "multiclass",
    "data": {
        "target": "vehicle_class",
    },
    "model": {
        "name": "lgbm",
        # --- Native LightGBM parameters ---
        "params": {
            "objective": "multiclass",
            "n_estimators": 1500,
            "learning_rate": 0.001,
            "max_depth": 5,
            "feature_fraction": 0.7,
            "bagging_fraction": 0.7,
            "bagging_freq": 10,
            "lambda_l2": 0.000001,
            "max_bin": 511,
        },
        # --- Smart parameters (resolved at fit time) ---
        "auto_num_leaves": True,
        "num_leaves_ratio": 1.0,
        "min_data_in_leaf_ratio": 0.01,
        "min_data_in_bin_ratio": 0.001,
        # balanced=True computes sample_weight from class frequencies
        "balanced": True,
    },
    "split": {
        "method": "stratified_kfold",
        "n_splits": 3,
    },
    "training": {
        "seed": 42,
        "early_stopping": {
            "enabled": True,
            "rounds": 150,
            "validation_ratio": 0.1,
        },
    },
    "evaluation": {
        "metrics": [
            "logloss",
            "auc",
            "auc_pr",
            "f1",
            "accuracy",
            "brier",
        ],
    },
}

## 4. Model Fit

In [ ]:
model = Model(config)
model.fit(data=df)
print("Fit complete.")

### 4.1 LightGBM Parameters

Config smart params and resolved booster params.

In [ ]:
model.params_table()

## 5. Evaluate

### 5.1 Metrics Table

In [ ]:
model.evaluate_table().round(4)

### 5.2 Learning Curve

In [ ]:
model.plot_learning_curve().show()

## 6. ROC Curve (OvR, IS vs OOS)

Multiclass ROC uses One-vs-Rest (OvR) strategy with per-class AUC.

In [ ]:
model.roc_curve_plot().show()

## 7. Confusion Matrix (IS vs OOS)

For multiclass, the confusion matrix is N×N (4×4 for 4 classes).

In [ ]:
cm = model.confusion_matrix()
print("=== OOS Confusion Matrix ===")
display(cm["oos"])
print("\n=== IS Confusion Matrix ===")
display(cm["is"])

## 8. Feature Importance: Split

In [ ]:
model.importance_plot(kind="split").show()

In [ ]:
pd.Series(model.importance(kind="split"), name="importance_split").sort_values(
    ascending=False
).to_frame()

## 9. Feature Importance: Gain

In [ ]:
model.importance_plot(kind="gain").show()

## 10. Feature Importance: SHAP

In [ ]:
model.importance_plot(kind="shap").show()

In [ ]:
pd.Series(model.importance(kind="shap"), name="mean_abs_shap").sort_values(
    ascending=False
).to_frame()